# OT Participant Tracking and Entity Chains

Follow named characters through OT narrative: what they do, what happens to them,
and where they appear in the story.

**Approach:**
- Primary: MACULA `participantref` column for precise entity resolution
- Fallback: lemma-based matching (covers the majority of cases)
- Co-occurrence heuristic: verbal tokens in the same verse as a participant

**Pre-identified participants (`KNOWN_OT_PARTICIPANTS`):**
YHWH, Elohim, Abraham, Isaac, Jacob, Joseph, Moses, Aaron, Joshua, David,
Solomon, Elijah, Isaiah, Jeremiah, Daniel, Ruth, Esther, Saul, Jonah

**Related notebooks:**
- `ot/speakers/speaker_attribution.ipynb` — who speaks in each book
- `ot/speakers/syntactic_roles_ot.ipynb` — what YHWH/Elohim does as subject

In [ ]:
# @title Colab setup (runs only on Google Colab)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess, os
    # Clone the repo so all source and data paths work
    if not os.path.isdir("/content/berean-bible-bots"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/dnovick/berean-bible-bots.git",
             "/content/berean-bible-bots"],
            check=True,
        )
    os.chdir("/content/berean-bible-bots")
    sys.path.insert(0, "/content/berean-bible-bots/src")
    # Install Python dependencies
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "binder/requirements.txt"],
        check=True,
    )
    # Download processed data files (~295 MB, one-time)
    subprocess.run(["bash", "binder/postBuild"], check=True)
    print("Colab environment ready.")


In [ ]:
import sys
sys.path.insert(0, '../../../src')

from bible_grammar import (
    KNOWN_OT_PARTICIPANTS,
    ot_participant_data, ot_participant_subject_verbs, ot_participant_object_verbs,
    ot_participant_chain, ot_entity_density, ot_participant_compare,
    print_ot_participant_profile, print_ot_participant_chain, print_ot_participant_compare,
    ot_participant_chain_chart, ot_entity_density_chart,
)
import pandas as pd

In [ ]:
# All pre-identified participants
pd.DataFrame([
    {'key': k, 'lemma': v['lemma'], 'gloss': v['gloss']}
    for k, v in KNOWN_OT_PARTICIPANTS.items()
])

## 1. Overview — What Participant Tracking Provides

Each participant profile shows:
- Total mentions across the OT
- Distribution by book
- Top verbal associations (same-verse co-occurrence)

In [ ]:
# YHWH — the most mentioned participant in the OT
print_ot_participant_profile('YHWH')

In [ ]:
# Moses — restricted to Torah
print_ot_participant_profile('Moses', book='Exo')

## 2. YHWH Participant Chain

Where does YHWH appear most densely in the OT narrative books?
High density often indicates divine intervention or judgment passages.

In [ ]:
# YHWH in Exodus — chapter by chapter
print_ot_participant_chain('Exo', 'YHWH')

In [ ]:
# YHWH in Isaiah — which half is more YHWH-dense?
yhwh_isa = ot_participant_chain('Isa', 'YHWH')
yhwh_isa['half'] = yhwh_isa['chapter'].apply(lambda c: '1–39' if c <= 39 else '40–66')
print("Isaiah YHWH mentions by half:")
print(yhwh_isa.groupby('half')['mention_count'].sum())

## 3. Patriarch Tracking — Abraham, Isaac, Jacob in Genesis

In [ ]:
# Compare the three patriarchs in Genesis
print_ot_participant_compare(['Abraham', 'Isaac', 'Jacob', 'Joseph'], book='Gen')

In [ ]:
# Visual: patriarchs by chapter in Genesis
ot_participant_chain_chart('Gen', ['Abraham', 'Isaac', 'Jacob', 'Joseph'])

In [ ]:
# Abraham's profile
print_ot_participant_profile('Abraham', book='Gen')

In [ ]:
# Jacob's chapter chain in Genesis
print_ot_participant_chain('Gen', 'Jacob')

## 4. Moses Across Exodus–Deuteronomy

In [ ]:
print_ot_participant_compare(['Moses', 'Aaron', 'YHWH'], book='Exo')

In [ ]:
ot_participant_chain_chart('Exo', ['Moses', 'Aaron', 'YHWH'])

In [ ]:
# Moses mentions across Torah
for book in ['Gen', 'Exo', 'Lev', 'Num', 'Deu']:
    data = ot_participant_data('Moses', book=book)
    print(f"{book}: {len(data):>4} mentions")

## 5. Comparison: Abraham vs. Moses

Both are foundational figures. How do their mention profiles and
verbal associations differ?

In [ ]:
print_ot_participant_compare(['Abraham', 'Moses', 'David', 'Solomon'])

In [ ]:
# Top verbs for Abraham
ab_verbs = ot_participant_subject_verbs('Abraham', book='Gen', top_n=15)
print("Abraham — top co-occurring verbs in Genesis:")
ab_verbs

In [ ]:
# Top verbs for Moses (Exodus)
mo_verbs = ot_participant_subject_verbs('Moses', book='Exo', top_n=15)
print("Moses — top co-occurring verbs in Exodus:")
mo_verbs

## 6. Entity Density Charts — Who Appears Where?

In [ ]:
# Genesis: all KNOWN_OT_PARTICIPANTS by chapter
ot_entity_density_chart('Gen')

In [ ]:
# 1 Samuel: David and Saul
ot_participant_chain_chart('1Sa', ['David', 'Saul', 'YHWH'])

In [ ]:
ot_entity_density_chart('1Sa')

## 7. Ad-hoc Queries

In [ ]:
# Jonah — a short book but dense with participant mentions
print_ot_participant_chain('Jon', 'Jonah')
ot_entity_density_chart('Jon')

In [ ]:
# Ruth mentions across the book of Ruth
print_ot_participant_chain('Rut', 'Ruth')

In [ ]:
# Ad-hoc: search by raw Hebrew lemma (not in KNOWN_OT_PARTICIPANTS)
# e.g., Boaz (בֹּעַז)
from bible_grammar import load_syntax_ot as load_ot_data
rut_df = load_ot_data()
boaz_tokens = rut_df[(rut_df['book'] == 'Rut') & (rut_df['lemma'] == 'בֹּעַז')]
print(f"Boaz mentions in Ruth: {len(boaz_tokens)}")
boaz_tokens.groupby('chapter').size().reset_index(name='count')